# Extra Trees Regressor

## 1. Setup and Data Loading

In [1]:
# 1) Imports 
import numpy as np
import pandas as pd

from datetime import datetime

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 2) Import pre processing helpers
#    -> this should define: full_train_dataset, cat_feat, num_feat,
#       basic_string_transformer, def_string_basic_transformer,
#       preprocess_categorical, MyTargetEncoder, MyOneHotEncoder, etc.
%run 05_0_preproc_helpers.ipynb  

# 3) Define target
TARGET_COL = "price"

# 4) Separate X and y from the treated dataset
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# Numerical and categorical features (splitted in pre processing)
numeric_features = num_feat
categorical_features = cat_feat

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)

X shape: (75973, 10)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


In this section we load all the preprocessing helpers and define the basic supervised learning setup for the ExtraTrees model.

- We import the required Python packages for model training, cross-validation, preprocessing and evaluation metrics.
- We run the preprocessing helper notebook (`05_0_preproc_helpers.ipynb`), which provides:
  - The cleaned training dataset (`full_train_dataset`);
  - The lists of numerical and categorical features (`num_feat` and `cat_feat`);
  - All custom preprocessing functions (numeric imputers, brand/model/gearbox/fuel resolvers, and the custom encoders `MyTargetEncoder` and `MyOneHotEncoder`).
- We define the target variable `price` and split the data into:
  - `X`: feature matrix;
  - `y`: target vector.
- Finally, we store the numeric and categorical feature names to ensure a consistent preprocessing pipeline in all subsequent steps.

This cell only prepares the data and helper objects. No modeling is performed yet.

## 2. Randomized Hyperparameter Search with K-Fold Cross-Validation

In [3]:
valid_transmissions = ["MANUAL", "AUTOMATIC", "SEMIAUTO"]
valid_fueltypes    = ["PETROL", "DIESEL", "HYBRID"]


N_SPLITS = 8
RANDOM_STATE = 42

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# HIPERPARAMETERS
# (ExtraTrees generally likes medium depth + high number of trees, but we can adjust this later)
param_distributions = {
    "n_estimators": [800, 1000, 1200],          # number of trees
    "max_depth": [None, 15, 20, 25],            # maximum depth of the tree 
    "min_samples_split": [3, 4, 6, 8],          # minimum samples to split an internal node
    "min_samples_leaf": [1, 2],                 # minimum samples at a leaf node
    "max_features": [0.3, 0.5, 0.6],            # number of features to consider at each split
    "bootstrap": [False],                       # ExtraTrees usually performs better without bootstrapping 
}

N_RANDOM_CONFIGS = 150  # changeable

sampler = ParameterSampler(
    param_distributions,
    n_iter=N_RANDOM_CONFIGS,
    random_state=RANDOM_STATE
)

search_results = []

# Best RMSE
best_rmse = np.inf
best_config_rmse = None

# Best MAE
best_mae = np.inf
best_config_mae = None

# Best combined score (0.5 RMSE + 0.5 MAE)
best_combo = np.inf
best_config_combo = None

log_path = "et_random_search_log.txt"

with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        """
        Keeps messages only on log file (keeping the notebook clean)
        """
        log_file.write(datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ") + msg + "\n")
        log_file.flush()

    log("# =============================")
    log("# START OF RANDOM SEARCH ExtraTrees")
    log("# =============================")
    log(f"N_SPLITS = {N_SPLITS}, N_RANDOM_CONFIGS = {N_RANDOM_CONFIGS}")
    log(f"param_distributions = {param_distributions}")

    # RANDOM SEARCH + CV
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########")
        log(f"Parameters: {params}")

        fold_rmses = []
        fold_maes  = []
        fold_r2s   = []

        fold_rmses_train = []
        fold_maes_train  = []
        fold_r2s_train   = []

        # CV for this configuration
        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

            log("")
            log(f"[CONFIG {config_id}] ==== FOLD {fold}/{N_SPLITS} ====")

            # 1) SEPARATE TRAIN / VALIDATION
            X_train = X.iloc[train_idx].copy()
            X_val   = X.iloc[val_idx].copy()
            y_train = y.iloc[train_idx].copy()
            y_val   = y.iloc[val_idx].copy()

            log(f"[C{config_id}|F{fold}] X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")
            log(f"[C{config_id}|F{fold}] y_train shape: {y_train.shape}, y_val shape: {y_val.shape}")

            # NaNs before numeric imputation
            log(f"[C{config_id}|F{fold}] NaNs before numeric imputation (only num features):")
            log(str(X_train[numeric_features].isna().sum()))

            log(f"[C{config_id}|F{fold}] NaNs before (categoricals):")
            log(str(X_train[categorical_features].isna().sum()))

            unknown_counts_before = (X_train[categorical_features] == "UNKNOWN").sum()
            log(f"[C{config_id}|F{fold}] 'UNKNOWN' before (categoricals):")
            log(str(unknown_counts_before))

            # 2) NUMERICAL PRE PROCESSING (fit for train, transform for val)
            year_state = fit_year_median(
                X_train,
                year_col="year",
                model_col="model"
            )
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            mileage_state = fit_mileage_imputer(
                X_train,
                mileage_col="mileage",
                do_abs=True
            )
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            engine_state = fit_engine_size_imputer(
                X_train,
                engine_col="engineSize"
            )
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            tax_state = fit_tax_imputer(
                X_train,
                tax_col="tax",
                do_abs=True
            )
            X_train = transform_tax_imputer(X_train, state=tax_state)
            X_val   = transform_tax_imputer(X_val,   state=tax_state)

            mpg_state = fit_mpg_imputer(
                X_train,
                mpg_col="mpg",
                do_abs=True
            )
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            '''paint_state = fit_paint_quality_imputer(
                X_train,
                paint_col="paintQuality%"
            )
            X_train = transform_paint_quality_imputer(X_train, state=paint_state)
            X_val   = transform_paint_quality_imputer(X_val,   state=paint_state)'''

            owners_state = fit_previous_owners_imputer(
                X_train,
                owners_col="previousOwners",
                year_col="year",
                mileage_col="mileage"
            )
            X_train = transform_previous_owners_imputer(X_train, state=owners_state)
            X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

            log(f"[C{config_id}|F{fold}] After numeric imputation: X_train shape = {X_train.shape}, X_val shape = {X_val.shape}")
            log(f"[C{config_id}|F{fold}] NaNs after numeric imputation (only num features):")
            log(str(X_train[numeric_features].isna().sum()))

            # CATEGORICALS
            # solve invalid / ambiguous brands 
            brand_state = fit_ambiguous_brand_resolver(
                train_df=X_train,
                valid_brands=valid_brands,
                brand_col="Brand",
                model_col="model",
                year_col="year",
            )

            X_train, brand_corr_train, brand_still_invalid_train = transform_ambiguous_brands(
                X_train,
                brand_state,
            )
            X_val, brand_corr_val, brand_still_invalid_val = transform_ambiguous_brands(
                X_val,
                brand_state,
            )

            log(f"[C{config_id}|F{fold}] After solving Brand:")
            log("  Nº still invalid brands (train): " + str(len(brand_still_invalid_train)))
            log("  Example of invalid brands (train): " + str(brand_still_invalid_train[:10]))

            # solve invalid / ambiguous models
            model_state = fit_invalid_model_resolver(
                train_df=X_train,
                valid_models_by_brand=valid_models_by_brand,
                brand_col="Brand",
                model_col="model",
                year_col="year",
                fuel_col="fuelType",
                mpg_col="mpg",
            )

            X_train, model_corr_train, model_still_invalid_train = transform_invalid_models(
                X_train,
                model_state,
            )
            X_val, model_corr_val, model_still_invalid_val = transform_invalid_models(
                X_val,
                model_state,
            )

            log(f"[C{config_id}|F{fold}] After solving model:")
            log("  Nº still invalid models (train): " + str(len(model_still_invalid_train)))
            log("  Example of invalid models (train): " + str(model_still_invalid_train[:10]))

            # solve transmission
            transm_state = fit_transmission_resolver(
                train_df=X_train,
                valid_transmissions=valid_transmissions,
                transm_col="transmission",
                brand_col="Brand",
                model_col="model",
                fuel_col="fuelType",
            )

            X_train, transm_corr_train, transm_still_invalid_train = transform_transmission_resolver(
                X_train,
                transm_state,
            )
            X_val, transm_corr_val, transm_still_invalid_val = transform_transmission_resolver(
                X_val,
                transm_state,
            )

            log(f"[C{config_id}|F{fold}] After solving transmission:")
            log("  Distinct values (train): " +
                str(sorted(X_train["transmission"].astype(str).str.upper().unique())))
            log("  Still problematic (train): " + str(transm_still_invalid_train[:10]))

            # solve fuelType
            fuel_state = fit_fueltype_resolver(
                train_df=X_train,
                valid_fueltypes=valid_fueltypes,
                fuel_col="fuelType",
                brand_col="Brand",
                model_col="model",
                transm_col="transmission",
            )

            X_train, fuel_corr_train, fuel_still_invalid_train = transform_fueltype_resolver(
                X_train,
                fuel_state,
            )
            X_val, fuel_corr_val, fuel_still_invalid_val = transform_fueltype_resolver(
                X_val,
                fuel_state,
            )

            log(f"[C{config_id}|F{fold}] After solving fuelType:")
            log("  Distinct values (train): " +
                str(sorted(X_train["fuelType"].astype(str).str.upper().unique())))
            log("  Still problematic (train): " + str(fuel_still_invalid_train[:10]))

            # NUMERICAL AND CATEGORY SUMMARY
            log(f"[C{config_id}|F{fold}] --- NUMERICAL (train, after imputation) ---")
            num_means_train = X_train[numeric_features].mean()
            num_nans_train  = X_train[numeric_features].isna().sum()
            log("  Means (train):")
            log(str(num_means_train))
            log("  NaNs (train):")
            log(str(num_nans_train))

            num_means_val = X_val[numeric_features].mean()
            num_nans_val  = X_val[numeric_features].isna().sum()
            log("  Means (val):")
            log(str(num_means_val))
            log("  NaNs  (val):")
            log(str(num_nans_val))

            log(f"[C{config_id}|F{fold}] --- CATEGORICAL (train, after solving every problematic entry) ---")
            cat_nans_train     = X_train[categorical_features].isna().sum()
            cat_unknown_train  = (X_train[categorical_features] == "UNKNOWN").sum()
            log("  NaNs (train):")
            log(str(cat_nans_train))
            log("  'UNKNOWN' (train):")
            log(str(cat_unknown_train))

            log(f"[C{config_id}|F{fold}] Distinct values (train):")
            for col in categorical_features:
                uniques = sorted(X_train[col].dropna().astype(str).unique())
                if len(uniques) > 30:
                    shown = uniques[:30]
                    log(f"    {col}: {len(uniques)} distinct values. First 30: {shown} ...")
                else:
                    log(f"    {col}: {len(uniques)} distinct values: {uniques}")

            # 3) CATEGORICAL ENCODING
            high_card_features = ["Brand", "model"]  # target encoding
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            log(f"[C{config_id}|F{fold}] high_card_features = {high_card_features}")
            log(f"[C{config_id}|F{fold}] low_card_features  = {low_card_features}")

            unknown_mask_train = (X_train[categorical_features] == "UNKNOWN")
            unknown_mask_val   = (X_val[categorical_features]   == "UNKNOWN")

            unknown_counts_train = unknown_mask_train.sum()
            unknown_counts_val   = unknown_mask_val.sum()

            rows_with_unknown_train = unknown_mask_train.any(axis=1).sum()
            rows_with_unknown_val   = unknown_mask_val.any(axis=1).sum()

            log(f"[C{config_id}|F{fold}] 'UNKNOWN' after solving (categoricals, train):")
            log(str(unknown_counts_train))
            log(f"[C{config_id}|F{fold}] 'UNKNOWN' after solving (categoricals, val):")
            log(str(unknown_counts_val))
            log(f"[C{config_id}|F{fold}] Nº observations with >=1 UNKNOWN (train): {rows_with_unknown_train}")
            log(f"[C{config_id}|F{fold}] Nº observations with >=1 UNKNOWN (val)  : {rows_with_unknown_val}")

            # 3.1) Target Encoding for brand and model
            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train)

            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            # 3.2) One-hot encoding for other categoricals
            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])

            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            log(f"[C{config_id}|F{fold}] X_train_cat shape: {X_train_cat.shape}")
            log(f"[C{config_id}|F{fold}] X_val_cat   shape: {X_val_cat.shape}")

            log(f"[C{config_id}|F{fold}] NaNs after encoding (X_train_cat, total): {X_train_cat.isna().sum().sum()}")
            log(f"[C{config_id}|F{fold}] NaNs after encoding (X_val_cat, total): {X_val_cat.isna().sum().sum()}")

            unknown_cols_train = [col for col in X_train_cat.columns if "UNKNOWN" in str(col)]
            unknown_cols_val   = [col for col in X_val_cat.columns if "UNKNOWN" in str(col)]

            log(f"[C{config_id}|F{fold}] Encoded columns with 'UNKNOWN' (train): {unknown_cols_train}")
            log(f"[C{config_id}|F{fold}] Encoded columns with 'UNKNOWN' (val)  : {unknown_cols_val}")

            # 4) NUMERIC SCALING
            scaler = StandardScaler()
            X_train_num = scaler.fit_transform(X_train[numeric_features])
            X_val_num   = scaler.transform(X_val[numeric_features])

            X_train_num_df = pd.DataFrame(
                X_train_num,
                index=X_train.index,
                columns=[f"num_{col}" for col in numeric_features]
            )
            X_val_num_df = pd.DataFrame(
                X_val_num,
                index=X_val.index,
                columns=[f"num_{col}" for col in numeric_features]
            )

            log(f"[C{config_id}|F{fold}] X_train_num_df shape: {X_train_num_df.shape}")
            log(f"[C{config_id}|F{fold}] X_val_num_df   shape: {X_val_num_df.shape}")

            # 5) FINAL MATRIX (NUM + CAT)
            X_train_final = pd.concat(
                [X_train_num_df, X_train_cat],
                axis=1
            )
            X_val_final = pd.concat(
                [X_val_num_df, X_val_cat],
                axis=1
            )

            log(f"[C{config_id}|F{fold}] X_train_final shape: {X_train_final.shape}")
            log(f"[C{config_id}|F{fold}] X_val_final   shape: {X_val_final.shape}")

            # --------------------------------------------------------------------------------
            # 6) EXTRA TREES REGRESSOR 
            et = ExtraTreesRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1,
                **params
            )

            log(f"[C{config_id}|F{fold}] Training ExtraTreesRegressor...")
            et.fit(X_train_final, y_train)
            
            log(f"[C{config_id}|F{fold}] Predicting...")

            y_pred_train = et.predict(X_train_final)
            y_pred_val   = et.predict(X_val_final)

            log(f"[C{config_id}|F{fold}] Predictions done.")

            # 7) FOLD METRICS (train and val)
            # VAL
            mse_val  = mean_squared_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mse_val)
            mae_val  = mean_absolute_error(y_val, y_pred_val)
            r2_val   = r2_score(y_val, y_pred_val)

            # TRAIN
            mse_tr  = mean_squared_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mse_tr)
            mae_tr  = mean_absolute_error(y_train, y_pred_train)
            r2_tr   = r2_score(y_train, y_pred_train)

            log(f"[C{config_id}|F{fold}] TRAIN -> RMSE: {rmse_tr:.4f} | MAE: {mae_tr:.4f} | R²: {r2_tr:.4f}")
            log(f"[C{config_id}|F{fold}] VAL   -> RMSE: {rmse_val:.4f} | MAE: {mae_val:.4f} | R²: {r2_val:.4f}")

            fold_rmses.append(rmse_val)
            fold_maes.append(mae_val)
            fold_r2s.append(r2_val)

            fold_rmses_train.append(rmse_tr)
            fold_maes_train.append(mae_tr)
            fold_r2s_train.append(r2_tr)

        # FOLDS MEAN FOR THIS CONFIGURATION
        mean_rmse_val = np.mean(fold_rmses)
        mean_mae_val  = np.mean(fold_maes)
        mean_r2_val   = np.mean(fold_r2s)

        mean_rmse_tr = np.mean(fold_rmses_train)
        mean_mae_tr  = np.mean(fold_maes_train)
        mean_r2_tr   = np.mean(fold_r2s_train)

        combo_score = 0.5 * mean_rmse_val + 0.5 * mean_mae_val

        log("")
        log(f"Config {config_id} - Avg TRAIN RMSE: {mean_rmse_tr:.4f} | MAE: {mean_mae_tr:.4f} | R²: {mean_r2_tr:.4f}")
        log(f"Config {config_id} - Avg VAL RMSE : {mean_rmse_val:.4f} | MAE: {mean_mae_val:.4f} | R²: {mean_r2_val:.4f}")
        log(f"Config {config_id} - Combined score (0.5*RMSE + 0.5*MAE) [VAL]: {combo_score:.4f}")

        search_results.append({
            "config_id": config_id,
            **params,
            "rmse_train_mean": mean_rmse_tr,
            "mae_train_mean": mean_mae_tr,
            "r2_train_mean": mean_r2_tr,
            "rmse_mean": mean_rmse_val,
            "mae_mean": mean_mae_val,
            "r2_mean": mean_r2_val,
            "combo_score": combo_score,
        })

        # keep best RMSE (val)
        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config_rmse = {**params}
            log(f"[NEW BEST RMSE] Config {config_id} with average RMSE (VAL) = {best_rmse:.4f}")

        # keep best MAE (val)
        if mean_mae_val < best_mae:
            best_mae = mean_mae_val
            best_config_mae = {**params}
            log(f"[NEW BEST MAE] Config {config_id} with average MAE (VAL) = {best_mae:.4f}")

        # keep best combined score
        if combo_score < best_combo:
            best_combo = combo_score
            best_config_combo = {**params}
            log(f"[NEW BEST COMBINED] Config {config_id} with score = {best_combo:.4f}")

    log("")
    log("# =============================")
    log("# END OF RANDOM SEARCH ExtraTrees")
    log("# =============================")
    log(f"Best configuration (min RMSE VAL): {best_config_rmse}")
    log(f"Best average RMSE (VAL): {best_rmse:.4f}")
    log(f"Best configuration (min MAE VAL): {best_config_mae}")
    log(f"Best average MAE  (VAL): {best_mae:.4f}")
    log(f"Best configuration (combined score VAL): {best_config_combo}")
    log(f"Best combined score (VAL): {best_combo:.4f}")

# -----------------------------------------------------------------------------
# RANDOM SEARCH'S FINAL SUMMARY 
results_df = pd.DataFrame(search_results)
results_df_sorted = results_df.sort_values(by="mae_mean", ascending=True)

display(results_df_sorted.head(10))

print("\nBest configuration found (min RMSE VAL):")
print(best_config_rmse)
print("Best average RMSE (VAL):", best_rmse)

print("\nBest configuration found (min MAE VAL):")
print(best_config_mae)
print("Best average MAE (VAL):", best_mae)

print("\nBest configuration found (VAL = 0.5*RMSE + 0.5*MAE):")
print(best_config_combo)
print("Best combined score (VAL):", best_combo)

print(f"\nDetailed logs in: {log_path}")

# Save results in CSV
results_df_sorted.to_csv("et_random_search_results.csv", index=False)

,config_id,n_estimators,min_samples_split,min_samples_leaf,max_features,max_depth,bootstrap,rmse_train_mean,mae_train_mean,r2_train_mean,rmse_mean,mae_mean,r2_mean,combo_score
20,21,800,4,1,0.6,25.0,False,966.302901,587.935307,0.990149,2181.723540,1274.330143,0.949588,1728.026841
119,120,1000,6,1,0.6,25.0,False,1198.952130,732.297715,0.984834,2184.743171,1275.746213,0.949461,1730.244692
103,104,1200,6,1,0.6,25.0,False,1199.038504,732.165079,0.984832,2184.787925,1275.818252,0.949460,1730.303088
94,95,1000,6,1,0.6,NaN,False,1098.100770,658.502752,0.987277,2190.138776,1276.034841,0.949208,1733.086808
14,15,1200,6,1,0.6,NaN,False,1098.145118,658.433227,0.987275,2189.577960,1276.077514,0.949235,1732.827737
106,107,1200,3,1,0.6,25.0,False,795.162581,473.658010,0.993330,2182.292976,1276.863869,0.949563,1729.578422
113,114,1000,3,1,0.6,25.0,False,795.464544,473.853528,0.993325,2182.404205,1277.005324,0.949557,1729.704764
90,91,1000,4,1,0.5,25.0,False,1007.114815,616.214900,0.989300,2194.738391,1279.971322,0.949001,1737.354856
100,101,800,8,1,0.6,NaN,False,1286.373415,772.865136,0.982541,2200.530820,1280.752319,0.948731,1740.641569
129,130,1200,3,1,0.5,25.0,False,834.979544,503.216607,0.992645,2196.795353,1282.246744,0.948904,1739.521048



Best configuration found (min RMSE VAL):
{'n_estimators': 800, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.6, 'max_depth': 25, 'bootstrap': False}
Best average RMSE (VAL): 2181.723539878467

Best configuration found (min MAE VAL):
{'n_estimators': 800, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.6, 'max_depth': 25, 'bootstrap': False}
Best average MAE (VAL): 1274.3301426780527

Best configuration found (VAL = 0.5*RMSE + 0.5*MAE):
{'n_estimators': 800, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.6, 'max_depth': 25, 'bootstrap': False}
Best combined score (VAL): 1728.0268412782598

Detailed logs in: et_random_search_log.txt


In this block we perform a random search over ExtraTreesRegressor hyperparameters using K-fold cross-validation (k=8), with all preprocessing done inside each fold to avoid data leakage.

Main steps:

1. **Cross-validation setup**  
   - We create an 8-fold K-fold object with shuffling and a fixed `random_state` to ensure reproducibility.
   - Each configuration will be evaluated across all 8 folds.

2. **Hyperparameter search space**  
   - We define a search space for:
     - `n_estimators` (number of trees),
     - `max_depth` (tree depth),
     - `min_samples_split` and `min_samples_leaf` (regularization via minimum samples),
     - `max_features` (proportion of features used at each split),
     - `bootstrap` and the tree split `criterion`.
   - We then sample a given number of random configurations (`N_RANDOM_CONFIGS`) from this space using `ParameterSampler`.

3. **Fold-level preprocessing (no leakage)**  
   For each configuration and each fold, we:
   - Split the data into `X_train`, `X_val`, `y_train`, `y_val`.
   - Apply the entire preprocessing pipeline **fit only on the training fold**:
     - Numeric imputations for `year`, `mileage`, `engineSize`, `tax`, `mpg`, `paintQuality%` and `previousOwners`;
     - Resolution of invalid or ambiguous categories for `Brand`, `model`, `transmission` and `fuelType`;
     - Target encoding for high-cardinality features (`Brand` and `model`);
     - One-hot encoding for the remaining categorical variables;
     - Scaling of all numeric variables with `StandardScaler`.
   - Concatenate the processed numeric and categorical features into a final design matrix for the model.

4. **Model training and evaluation per fold**  
   - For each fold we train an `ExtraTreesRegressor` instance with the current hyperparameters.
   - We compute performance metrics on both training and validation sets:
     - RMSE, MAE and R².
   - We log all intermediate checks (NaNs, UNKNOWN values, distinct categories, shapes) into a text file so we can verify that the preprocessing is stable.

5. **Aggregating results by configuration**  
   - For each configuration we average the RMSE, MAE and R² across folds.
   - We keep track of:
     - The configuration with the lowest validation RMSE;
     - The configuration with the lowest validation MAE;
     - The configuration with the best combined score (a weighted average of RMSE and MAE).
   - All results are stored in a DataFrame and saved to CSV for later analysis and reporting.

By the end of this cell we have a robust estimate of how different ExtraTrees configurations perform under cross-validation, and we select the best hyperparameters (in terms of RMSE) to train the final model on the full dataset.

## 3. Final Model Training and Kaggle Submission Generation

In [ ]:
# 0) FINAL CONFIGURATION
# Using the best configuration considering RMSE found in the Random Search (cell above)
final_config_et = best_config_rmse
print("Final config ExtraTrees used for train:", final_config_et)


# 1) PREPARE TRAIN FEATURES
X_full = X.copy()
y_full = y.copy()

# a) STRING NORMALIZATION (apply to train)
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

for col in cols_to_normalize:
    if col in X_full.columns:
        # Using apply/lambda for safety with your custom function
        X_full[col] = X_full[col].apply(
            lambda x: basic_string_transformer(
                x, 
                remove_middle_spaces=True, 
                allow_extra_chars=""
            )
        )

# Define feature sets
high_card_features = ["Brand", "model"] 
low_card_features  = [c for c in categorical_features if c not in high_card_features]


# 2) NUMERIC PRE PROCESSING - TRAIN (FIT & TRANSFORM)
# Year
year_state = fit_year_median(X_full, year_col="year", model_col="model")
X_full = transform_year_with_model_median(X_full, state=year_state)

# Mileage
mileage_state = fit_mileage_imputer(X_full, mileage_col="mileage", do_abs=True)
X_full = transform_mileage_imputer(X_full, state=mileage_state)

# Engine Size
engine_state = fit_engine_size_imputer(X_full, engine_col="engineSize")
X_full = transform_engine_size_imputer(X_full, state=engine_state)

# Tax
tax_state = fit_tax_imputer(X_full, tax_col="tax", do_abs=True)
X_full = transform_tax_imputer(X_full, state=tax_state)

# MPG
mpg_state = fit_mpg_imputer(X_full, mpg_col="mpg", do_abs=True)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

# Paint Quality 
# paint_state = fit_paint_quality_imputer(X_full, paint_col="paintQuality%")
# X_full = transform_paint_quality_imputer(X_full, state=paint_state)

# Previous Owners
owners_state = fit_previous_owners_imputer(
    X_full, owners_col="previousOwners", year_col="year", mileage_col="mileage"
)
X_full = transform_previous_owners_imputer(X_full, state=owners_state)


# 3) CATEGORICAL RESOLVERS - TRAIN (FIT & TRANSFORM)
# Brand
brand_state = fit_ambiguous_brand_resolver(
    train_df=X_full, valid_brands=valid_brands, 
    brand_col="Brand", model_col="model", year_col="year"
)
X_full, _, brand_still_invalid_full = transform_ambiguous_brands(X_full, brand_state)
print(f"Train - Invalid Brands remaining: {len(brand_still_invalid_full)}")

# Model
model_state = fit_invalid_model_resolver(
    train_df=X_full, valid_models_by_brand=valid_models_by_brand,
    brand_col="Brand", model_col="model", year_col="year", 
    fuel_col="fuelType", mpg_col="mpg"
)
X_full, _, model_still_invalid_full = transform_invalid_models(X_full, model_state)
print(f"Train - Invalid Models remaining: {len(model_still_invalid_full)}")

# Transmission
transm_state = fit_transmission_resolver(
    train_df=X_full, valid_transmissions=valid_transmissions,
    transm_col="transmission", brand_col="Brand", 
    model_col="model", fuel_col="fuelType"
)
X_full, _, _ = transform_transmission_resolver(X_full, transm_state)

# Fuel Type
fuel_state = fit_fueltype_resolver(
    train_df=X_full, valid_fueltypes=valid_fueltypes,
    fuel_col="fuelType", brand_col="Brand", 
    model_col="model", transm_col="transmission"
)
X_full, _, _ = transform_fueltype_resolver(X_full, fuel_state)


# 4) CATEGORICAL ENCODING - TRAIN (FIT & TRANSFORM)
# Target Encoding
te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full)
X_full_high = te.transform(X_full[high_card_features])

# One-Hot Encoding
ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_features])
X_full_low = ohe.transform(X_full[low_card_features])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1)


# 5) NUMERIC SCALING - TRAIN (FIT & TRANSFORM)
scaler = StandardScaler()
X_full_num = scaler.fit_transform(X_full[numeric_features])

X_full_num_df = pd.DataFrame(
    X_full_num,
    index=X_full.index,
    columns=[f"num_{col}" for col in numeric_features]
)


# 6) FINAL MATRIX - TRAIN
X_full_final = pd.concat([X_full_num_df, X_full_cat], axis=1)
print("X_full_final shape:", X_full_final.shape)


# 7) TRAIN MODEL
et_final = ExtraTreesRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **final_config_et
)

print("Training final ExtraTrees model...")
et_final.fit(X_full_final, y_full)
print("Done.")


# 8) PREPARE TEST FEATURES
test_df = pd.read_csv("../../project_data/test.csv")

# a) STRING NORMALIZATION (apply to test)
# MUST be identical to train step
for col in cols_to_normalize:
    if col in test_df.columns:
        test_df[col] = test_df[col].apply(
            lambda x: basic_string_transformer(
                x, 
                remove_middle_spaces=True, 
                allow_extra_chars=""
            )
        )

X_test = test_df[numeric_features + categorical_features].copy()

# b) NUMERIC PREPROCESSING - TEST (TRANSFORM ONLY)
X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_mileage_imputer(X_test, state=mileage_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_imputer(X_test, state=tax_state)
X_test = transform_mpg_imputer(X_test, state=mpg_state)
# X_test = transform_paint_quality_imputer(X_test, state=paint_state) 
X_test = transform_previous_owners_imputer(X_test, state=owners_state)

# c) CATEGORICAL RESOLVERS - TEST (TRANSFORM ONLY)
X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

# d) ENCODING - TEST (TRANSFORM ONLY)
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_features])
X_test_cat  = pd.concat([X_test_high, X_test_low], axis=1)

# e) SCALING - TEST (TRANSFORM ONLY)
X_test_num = scaler.transform(X_test[numeric_features])
X_test_num_df = pd.DataFrame(
    X_test_num,
    index=X_test.index,
    columns=[f"num_{col}" for col in numeric_features]
)


# 9) FINAL MATRIX & PREDICTION
X_test_final = pd.concat([X_test_num_df, X_test_cat], axis=1)

# ensure test columns are in the EXACT same order as train columns - this prevents errors if OneHotEncoder outputs columns in a different order
X_test_final = X_test_final[X_full_final.columns]

print("X_test_final shape:", X_test_final.shape)

y_test_pred = et_final.predict(X_test_final)

# Summary
print("Predictions summary:")
print(pd.Series(y_test_pred).describe())

# Prepare Submission
y_test_pred_round = np.round(y_test_pred).astype(int)

submission = pd.DataFrame({
    "carID": test_df["carID"].astype(int),
    "price": y_test_pred_round
})

# Dynamic filename generation (reflecting the parameters used)
p = final_config_et
sub_name = f"et_final_submission_n{p['n_estimators']}_d{p['max_depth']}_feat{p['max_features']}.csv"

submission.to_csv(sub_name, index=False)
print(f"Submission file created: {sub_name}")

Final config ExtraTrees used for train: {'n_estimators': 800, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.6, 'max_depth': 25, 'bootstrap': False}
Train - Invalid Brands remaining: 0
Train - Invalid Models remaining: 1
X_full_final shape: (75973, 16)
Training final ExtraTrees model...
Done.
X_test_final shape: (32567, 16)
Predictions summary:
count     32567.000000
mean      16921.056262
std        9429.121160
min        1385.122396
25%       10412.671782
50%       14690.186750
75%       20894.880709
max      137184.666250
dtype: float64
Submission file created: et_final_submission_n800_d25_feat0.6.csv
